# ta_patterns — an end-to-end walkthrough

`ta_patterns` is a pure-NumPy library of **300 technical-analysis pattern
detectors**: 106 candlestick patterns and 194 chart patterns (short-bar,
double/multi, classic-geometric, harmonic, volume, busted).

Every detector has the same shape of contract:

- it takes OHLC(V) arrays — NumPy arrays or pandas Series, the index is stripped
- it returns a signed **`int8` array of the same length**: `+1` bullish,
  `-1` bearish, `0` no signal
- it is **point-in-time**: the value at bar *i* depends only on bars `0..i`,
  so a column can be dropped straight into a backtest without look-ahead

This notebook runs the library over real daily equity data and shows what you
actually get back: single detectors, the full 300-column signal matrix, the
aggregate score, and end-to-end timings.

Requires `yfinance` and `pandas` for the data step (`pip install yfinance`).
With no network, use the synthetic series defined at the end of section 1 —
every later cell works on it unchanged.

In [1]:
import inspect
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("../src"))    # run from a source checkout

import ta_patterns as tap
import ta_patterns.chart_patterns as cp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)

print(f"ta_patterns {tap.__version__}  |  numpy {np.__version__}  |  "
      f"python {sys.version.split()[0]}")
print(f"{len(tap.list_all_patterns())} detectors: "
      f"{len(tap.list_all_patterns(module='candles'))} candlestick + "
      f"{len(tap.list_all_patterns(module='chart'))} chart")

ta_patterns 1.1.1  |  numpy 2.1.2  |  python 3.12.7
300 detectors: 106 candlestick + 194 chart


## 1. Getting data

Five years of daily bars for four symbols, cached to `_data/` as CSV so the
notebook re-runs without hitting the network.

In [2]:
import yfinance as yf

TICKERS = ["AAPL", "MSFT", "SPY", "TSLA"]
PERIOD = "5y"


def load(ticker, period=PERIOD):
    path = f"_data/{ticker}_{period}.csv"
    if os.path.exists(path):
        frame = pd.read_csv(path, index_col=0)
    else:
        raw = yf.Ticker(ticker).history(period=period, interval="1d",
                                        auto_adjust=True)
        frame = raw[["Open", "High", "Low", "Close", "Volume"]].dropna()
        frame.columns = ["open", "high", "low", "close", "volume"]
        os.makedirs("_data", exist_ok=True)
        frame.to_csv(path)

    # Daily bars carry an exchange timezone that survives the CSV round-trip as
    # mixed EST/EDT offsets, i.e. an object-dtype index.  Drop it so the index
    # is a real DatetimeIndex and prints as plain dates.
    frame.index = pd.to_datetime(frame.index, utc=True).tz_localize(None).normalize()
    frame.index.name = "date"
    return frame


frames = {t: load(t) for t in TICKERS}
df = frames["AAPL"]
o, h, l, c, v = df.open, df.high, df.low, df.close, df.volume

print(f"AAPL: {len(df)} bars, {df.index[0].date()} .. {df.index[-1].date()}")
df.tail(3)

AAPL: 1254 bars, 2021-09-22 .. 2026-09-21


,open,high,low,close,volume
date,,,,,
2026-09-17,334.769989,338.339996,330.179993,337.000000,36700200
2026-09-18,337.910004,338.489990,332.529999,336.130005,86588200
2026-09-21,335.279999,339.640015,333.049988,338.980011,34955500


No network? Every cell below works on this synthetic series instead — assign it
to `df` and re-derive `o, h, l, c, v`. It is also what the benchmark section
uses for series longer than the real history.

In [3]:
def synthetic(n=2600, seed=0):
    """Deterministic GBM series with a realistic intrabar range."""
    rng = np.random.default_rng(seed)
    close = 100 * np.exp(np.cumsum(rng.normal(0.0003, 0.013, n)))
    open_ = close * (1 + rng.normal(0, 0.004, n))
    high = np.maximum(open_, close) * (1 + np.abs(rng.normal(0, 0.006, n)))
    low = np.minimum(open_, close) * (1 - np.abs(rng.normal(0, 0.006, n)))
    vol = rng.integers(int(2e6), int(9e7), n).astype(float)
    return pd.DataFrame({"open": open_, "high": high, "low": low,
                         "close": close, "volume": vol},
                        index=pd.bdate_range("2015-01-01", periods=n))


synthetic(5).round(2)

,open,high,low,close,volume
2015-01-01,100.34,100.71,99.75,100.19,37484986.0
2015-01-02,100.57,100.60,99.73,100.05,4492131.0
2015-01-05,101.30,102.71,100.73,100.92,2471032.0
2015-01-06,100.80,101.22,100.55,101.09,12936928.0
2015-01-07,99.91,101.17,99.28,100.42,2729019.0


## 2. One detector

`tap.hammer` is a single-bar candlestick pattern: a small body with a long
lower shadow, after a downtrend. Like every detector it returns an `int8` array
the same length as the input.

In [4]:
sig = tap.hammer(o, h, l, c)

print(f"returns {type(sig).__name__}, dtype={sig.dtype}, shape={sig.shape}")
print(f"fired on {int((sig != 0).sum())} of {len(sig)} bars "
      f"({(sig != 0).mean():.2%})")

hammer_dates = df.index[sig == 1]
print(f"last 5 hammers: {[str(d.date()) for d in hammer_dates[-5:]]}")

returns ndarray, dtype=int8, shape=(1254,)
fired on 21 of 1254 bars (1.67%)
last 5 hammers: ['2025-01-13', '2025-01-21', '2025-12-11', '2025-12-18', '2026-03-06']


Because the output is just an aligned array, putting it next to the prices is a
column assignment. Here are the bars around the most recent hammer, with the
5-bar forward return for context.

In [5]:
i = df.index.get_loc(hammer_dates[-1])
around = df.iloc[i - 2:i + 4][["open", "high", "low", "close"]].round(2)
around["hammer"] = sig[i - 2:i + 4]
around["fwd_5d_%"] = (df.close.pct_change(5).shift(-5) * 100).iloc[i - 2:i + 4].round(2)
around

,open,high,low,close,hammer,fwd_5d_%
date,,,,,,
2026-03-04,264.18,265.68,260.95,262.05,0,-0.65
2026-03-05,260.33,261.09,256.79,259.83,0,-1.74
2026-03-06,258.17,258.31,253.92,257.00,1,-2.85
2026-03-09,255.23,260.68,253.23,259.42,0,-2.72
2026-03-10,257.19,262.01,256.49,260.37,0,-2.53
2026-03-11,260.62,261.66,259.09,260.35,0,-4.17


## 3. Every threshold is a keyword argument

Detectors ship with strict textbook defaults. Nothing is hard-coded — the ratios
that define each shape are keyword arguments, so you can trade precision for
recall.

In [6]:
pd.DataFrame([
    {"arguments": "(defaults)",
     "fires": int((tap.hammer(o, h, l, c) != 0).sum())},
    {"arguments": "shadow_factor=1.5",
     "fires": int((tap.hammer(o, h, l, c, shadow_factor=1.5) != 0).sum())},
    {"arguments": "require_trend=False",
     "fires": int((tap.hammer(o, h, l, c, require_trend=False) != 0).sum())},
    {"arguments": "both",
     "fires": int((tap.hammer(o, h, l, c, shadow_factor=1.5,
                              require_trend=False) != 0).sum())},
]).set_index("arguments").rename_axis("tap.hammer(o, h, l, c, ...)")

,fires
"tap.hammer(o, h, l, c, ...)",
(defaults),21
shadow_factor=1.5,21
require_trend=False,37
both,39


The trend gate is doing most of the work here — a longer shadow alone changes
nothing on this symbol, but dropping the preceding-downtrend requirement nearly
doubles the hits.

For the pivot-based chart patterns the two knobs that matter most are `pivot_n`
(the neighbourhood half-width that qualifies a swing — larger means fewer,
more significant pivots) and the pattern's own shape tolerance. For
`double_top`, `tol` is how closely the two peaks must match in price.

In [7]:
grid = pd.DataFrame(
    [[int((cp.double_top(o, h, l, c, pivot_n=pivot_n, tol=tol) != 0).sum())
      for tol in (0.01, 0.03, 0.06)]
     for pivot_n in (3, 5, 7)],
    index=pd.Index([3, 5, 7], name="pivot_n"),
    columns=pd.Index(["tol=1%", "tol=3% (default)", "tol=6%"], name="fires"),
)
grid

fires,tol=1%,tol=3% (default),tol=6%
pivot_n,,,
3,52,92,176
5,45,79,112
7,7,78,119


Strict defaults can also legitimately find *nothing*. `ascending_triangle`
needs a flat resistance line with a rising support line; a strongly trending
symbol like AAPL never presents one, so it reports zero however you tune the
pivots. That is the detector being correct, not broken — worth remembering
before you go hunting for a bug.

## 4. `forming` vs `confirmed`

Chart patterns take `mode`. `'forming'` fires as soon as the shape is complete;
`'confirmed'` (the default) waits for the breakout that validates it. Confirmed
therefore fires later and less often — and is the honest choice for a backtest.

In [8]:
rows = []
for name in ["double_top", "double_bottom", "hs_top", "hs_bottom",
             "rounding_bottom", "cup_with_handle", "ascending_triangle",
             "descending_triangle"]:
    fn = getattr(cp, name)
    forming = int((fn(o, h, l, c, mode="forming") != 0).sum())
    confirmed = int((fn(o, h, l, c, mode="confirmed") != 0).sum())
    rows.append({"pattern": name, "forming": forming, "confirmed": confirmed,
                 "confirmed/forming": (f"{confirmed / forming:.0%}"
                                       if forming else "-")})

pd.DataFrame(rows).set_index("pattern")

,forming,confirmed,confirmed/forming
pattern,,,
double_top,356,79,22%
double_bottom,346,139,40%
hs_top,18,8,44%
hs_bottom,11,10,91%
rounding_bottom,19,13,68%
cup_with_handle,652,309,47%
ascending_triangle,0,0,-
descending_triangle,0,0,-


In [9]:
# Where the confirmed head-and-shoulders tops actually landed
hs = cp.hs_top(o, h, l, c, mode="confirmed")
pd.DataFrame({"close": df.close[hs != 0].round(2),
              "hs_top": hs[hs != 0]}).tail(5)

,close,hs_top
date,,
2024-03-05,168.29,-1
2024-09-04,219.02,-1
2025-04-08,171.37,-1
2026-01-12,259.54,-1
2026-03-13,249.67,-1


## 5. The full signal matrix

`batch_all` runs every candlestick and chart detector and returns one `int8`
column per pattern — the primary entry point for feature engineering.

In [10]:
tap.clear_cache()
t0 = time.perf_counter()
X = tap.batch_all(o, h, l, c, v=v)
elapsed = time.perf_counter() - t0

X.index = df.index            # batch_all returns a bare RangeIndex

print(f"{X.shape[0]} bars x {X.shape[1]} patterns in {elapsed * 1000:.0f} ms")
print(f"dtypes: {set(map(str, X.dtypes))}")
print(f"memory: {X.memory_usage(deep=True).sum() / 1e6:.2f} MB "
      f"for {X.size:,} cells")

counts = pd.Series(X.to_numpy().ravel()).value_counts().sort_index()
print("values: " + ", ".join(f"{int(k):>2d}: {int(n):,}" for k, n in counts.items()))

fired = X.loc[:, (X != 0).any()]
print(f"{fired.shape[1]} of {X.shape[1]} patterns fired at least once; "
      f"{X.shape[1] - fired.shape[1]} never fired")

1254 bars x 300 patterns in 169 ms
dtypes: {'int8'}
memory: 0.42 MB for 376,200 cells
values: -1: 8,782,  0: 355,821,  1: 11,597
253 of 300 patterns fired at least once; 47 never fired


A real slice — the last few bars against the columns that fire most often on
this symbol:

In [11]:
busiest = (fired != 0).sum().sort_values(ascending=False).head(6).index
pd.concat([df[["close"]].round(2), X[busiest]], axis=1).tail(8)

,close,white_candle,two_did,black_candle,falling_volume_trend,rising_volume_trend,two_tall
date,,,,,,,
2026-09-10,326.57,1,1,0,0,1,1
2026-09-11,332.27,1,0,0,0,1,0
2026-09-14,333.08,0,0,-1,0,1,0
2026-09-15,331.34,1,0,0,0,1,0
2026-09-16,332.41,0,0,-1,0,1,0
2026-09-17,337.00,1,1,0,0,1,1
2026-09-18,336.13,0,-1,-1,0,1,0
2026-09-21,338.98,1,1,0,0,1,1


Only ~5% of cells are non-zero, but that still averages out to a double-digit
number of simultaneous signals per bar across 300 detectors. The frequency is
very uneven — a handful of simple candlestick shapes dominate.

In [12]:
freq = pd.DataFrame({
    "fires": (X != 0).sum(),
    "bullish": (X == 1).sum(),
    "bearish": (X == -1).sum(),
})
freq["% of bars"] = (freq.fires / len(X) * 100).round(1)
freq.sort_values("fires", ascending=False).head(12)

,fires,bullish,bearish,% of bars
white_candle,684,684,0,54.5
two_did,618,301,317,49.3
black_candle,568,0,568,45.3
falling_volume_trend,552,0,552,44.0
rising_volume_trend,526,526,0,41.9
two_tall,491,266,225,39.2
flat_base,428,428,0,34.1
one_two_three,418,211,207,33.3
two_dance,381,206,175,30.4
two_close,346,171,175,27.6


For a dashboard, the useful query is usually "what is firing right now".

In [13]:
last = X.iloc[-1]
active = last[last != 0]

print(f"{df.index[-1].date()}  close={df.close.iloc[-1]:.2f}  "
      f"-> {len(active)} active signals\n")
for name, val in active.items():
    print(f"  {'+1 bullish' if val == 1 else '-1 bearish'}  {name}")

2026-09-21  close=338.98  -> 13 active signals

  +1 bullish  engulfing_bullish
  +1 bullish  white_candle
  +1 bullish  double_bottom_ugly
  +1 bullish  hook_reversal
  +1 bullish  hook_reversal_bottom
  +1 bullish  rising_volume_trend
  +1 bullish  three_valleys
  +1 bullish  two_close
  +1 bullish  two_close_bullish
  +1 bullish  two_did
  +1 bullish  two_did_bullish
  +1 bullish  two_tall
  +1 bullish  two_tall_bullish


`signals_at` is a convenience view over the candlestick half, returning
`(name, direction)` pairs and labelling the non-directional shapes (plain doji,
high wave) that carry no bull/bear bias.

In [14]:
for idx in (-1, -2, -3):
    pairs = tap.signals_at(o, h, l, c, idx)
    print(f"{df.index[idx].date()}:")
    for name, direction in pairs:
        print(f"    {name:26s} {direction}")

2026-09-21:
    engulfing_bullish          bullish
    white_candle               bullish
2026-09-18:
    black_candle               bearish
    hanging_man                bearish
    short_black_candle         non_directional
    tweezer_tops               bearish
2026-09-17:
    short_white_candle         non_directional
    spinning_top_white         bullish
    white_candle               bullish


## 6. The aggregate score

`net_score_all` sums the directional patterns into one `int16` series. Combined
detectors (`two_b`, `key_reversal`, …) and non-directional and bidirectional
patterns are excluded, so the score stays sign-balanced and nothing is
double-counted.

In [15]:
score = pd.Series(tap.net_score_all(o, h, l, c, v=v), index=df.index,
                  name="net_score")

print(f"dtype {score.dtype}, range [{score.min()}, {score.max()}], "
      f"non-zero on {(score != 0).mean():.1%} of bars")

bull, bear = score.nlargest(5), score.nsmallest(5)

pd.DataFrame({
    "most bullish": bull.index.strftime("%Y-%m-%d"),
    "score": bull.to_numpy(),
    "most bearish": bear.index.strftime("%Y-%m-%d"),
    "score ": bear.to_numpy(),
})

dtype int16, range [-17, 23], non-zero on 96.6% of bars


,most bullish,score,most bearish,score
0,2024-06-11,23,2023-09-20,-17
1,2023-05-05,21,2022-07-18,-16
2,2022-10-28,19,2024-04-01,-16
3,2024-06-24,19,2022-04-26,-15
4,2025-08-06,19,2023-01-03,-15


Bucketing the forward return by score is the obvious first sanity check. On one
symbol over five years this is descriptive only — no costs, no significance
test, and the buckets are wildly unbalanced. Do not read a strategy into it.

In [16]:
fwd = df.close.pct_change(10).shift(-10) * 100
buckets = pd.cut(score, [-99, -3, -1, 0, 1, 3, 99],
                 labels=["<=-3", "-2..-1", "0", "+1", "+2..+3", ">=+3"])

summary = fwd.groupby(buckets, observed=True).agg(["count", "mean", "std"]).round(2)
summary.columns = ["bars", "mean fwd 10d %", "std"]
summary

,bars,mean fwd 10d %,std
net_score,,,
<=-3,397,0.93,5.62
-2..-1,116,0.90,5.17
0,42,2.39,5.14
+1,46,1.61,5.15
+2..+3,135,0.54,5.46
>=+3,508,0.63,5.11


## 7. Across several symbols

Nothing is stateful, so scanning a universe is a loop. `clear_cache()` before
each symbol makes the timings comparable rather than measuring cache hits.

In [17]:
rows = []
for ticker, frame in frames.items():
    a = (frame.open, frame.high, frame.low, frame.close, frame.volume)
    tap.clear_cache()
    t0 = time.perf_counter()
    sigs = tap.scan_all_patterns(*a[:4], v=a[4])
    ms = (time.perf_counter() - t0) * 1000

    M = np.vstack([sigs[k] for k in sorted(sigs)])
    on_last_bar = [k for k in sorted(sigs) if sigs[k][-1] != 0]
    rows.append({
        "bars": len(frame),
        "scan ms": round(ms),
        "patterns fired": int((M != 0).any(axis=1).sum()),
        "signals": int((M != 0).sum()),
        "signals/bar": round((M != 0).sum() / len(frame), 1),
        "density": f"{(M != 0).mean():.2%}",
        "on last bar": len(on_last_bar),
    })

pd.DataFrame(rows, index=list(frames)).rename_axis("ticker")

,bars,scan ms,patterns fired,signals,signals/bar,density,on last bar
ticker,,,,,,,
AAPL,1254,168,253,20379,16.3,5.42%,13
MSFT,1254,167,254,20253,16.2,5.38%,20
SPY,1254,168,251,21471,17.1,5.71%,22
TSLA,1254,168,250,20912,16.7,5.56%,16


## 8. Benchmarks

Timings below are single-core, best-of-three, measured on whatever machine
executed this notebook. Re-run it to get figures for your own.

In [18]:
def timed(fn):
    t0 = time.perf_counter()
    fn()
    return time.perf_counter() - t0


def best_of(fn, runs=3):
    fn()                                  # warm up
    return min(timed(fn) for _ in range(runs))

### Entry points compared

The candlestick half is nearly free; essentially all the cost is the 194 chart
patterns and their pivot geometry.

In [19]:
entry_points = [
    ("scan_all — 106 candlesticks", lambda: tap.scan_all(o, h, l, c)),
    ("chart_scan_all — 194 chart", lambda: cp.chart_scan_all(o, h, l, c, v=v)),
    ("scan_all_patterns — all 300", lambda: tap.scan_all_patterns(o, h, l, c, v=v)),
    ("net_score_all — all 300", lambda: tap.net_score_all(o, h, l, c, v=v)),
    ("batch_all — all 300, DataFrame", lambda: tap.batch_all(o, h, l, c, v=v)),
]

pd.DataFrame(
    [{"ms": round(best_of(lambda fn=fn: (tap.clear_cache(), fn())) * 1000)}
     for _, fn in entry_points],
    index=[label for label, _ in entry_points],
).rename_axis(f"{len(df)} bars")

,ms
1254 bars,
scan_all — 106 candlesticks,2
chart_scan_all — 194 chart,167
scan_all_patterns — all 300,170
net_score_all — all 300,170
"batch_all — all 300, DataFrame",172


### Scaling

Real history first, then synthetic series for the lengths a daily feed cannot
reach. Note that `ms / 1k bars` is **not** flat — throughput degrades on long
series. Section 9 finds out why.

In [20]:
rows = []
for n in (250, 500, 1000, len(df)):
    a = df.iloc[-n:]
    ms = best_of(lambda a=a: (tap.clear_cache(),
                              tap.scan_all_patterns(a.open, a.high, a.low,
                                                    a.close, v=a.volume))) * 1000
    rows.append({"bars": n, "source": "real", "ms": round(ms),
                 "ms / 1k bars": round(ms / n * 1000, 1),
                 "bars/sec": f"{n / ms * 1000:,.0f}"})

for n in (5_000, 10_000):
    a = synthetic(n, seed=42)
    ms = best_of(lambda a=a: (tap.clear_cache(),
                              tap.scan_all_patterns(a.open, a.high, a.low,
                                                    a.close, v=a.volume)),
                 runs=1) * 1000
    rows.append({"bars": n, "source": "synthetic", "ms": round(ms),
                 "ms / 1k bars": round(ms / n * 1000, 1),
                 "bars/sec": f"{n / ms * 1000:,.0f}"})

pd.DataFrame(rows).set_index("bars")

,source,ms,ms / 1k bars,bars/sec
bars,,,,
250,real,36,144.7,"6,911"
500,real,70,139.8,"7,153"
1000,real,141,141.4,"7,071"
1254,real,175,139.5,"7,170"
5000,synthetic,750,149.9,"6,671"
10000,synthetic,1810,181.0,"5,525"


### The feature cache

Pivot and trendline results are memoised on array *contents*, so detectors that
ask for the same feature share one computation. The scanner already shares
pivots within a single call, so the cache mainly pays off on repeated scans of
the same arrays — a modest win, not a transformative one.

It is bounded on both entry count and payload bytes, and can be switched off
with `TA_PATTERNS_NO_CACHE=1`.

In [21]:
from ta_patterns.chart_patterns import _memo

rows = []
for n, frame in ((len(df), df), (4_000, synthetic(4_000, seed=42))):
    a = frame
    scan = lambda a=a: tap.scan_all_patterns(a.open, a.high, a.low, a.close,
                                             v=a.volume)
    runs = 3 if n <= 2000 else 1

    cold = best_of(lambda: (tap.clear_cache(), scan()), runs=runs) * 1000
    warm = best_of(scan, runs=runs) * 1000

    _memo._ENABLED = False          # same effect as TA_PATTERNS_NO_CACHE=1
    off = best_of(scan, runs=runs) * 1000
    _memo._ENABLED = True

    tap.clear_cache()
    scan()
    info = tap.cache_info()
    rows.append({"bars": n, "cache off ms": round(off), "cold ms": round(cold),
                 "warm ms": round(warm),
                 "warm speedup": f"{off / warm:.2f}x",
                 "entries": info["entries"],
                 "MB": round(info["nbytes"] / 1e6, 2)})

print(f"caps: {info['maxsize']} entries / {info['maxbytes'] / 1e6:.0f} MB")
pd.DataFrame(rows).set_index("bars")

caps: 64 entries / 67 MB


,cache off ms,cold ms,warm ms,warm speedup,entries,MB
bars,,,,,,
1254,184,171,170,1.08x,9,0.13
4000,620,591,589,1.05x,9,0.42


### Cost per detector

Signatures vary — `black_candle(o, c)`, `hammer(o, h, l, c)`,
`rising_volume_trend(o, h, l, c, v)` — so bind arguments by parameter name.

In [22]:
arrays = {"o": o, "h": h, "l": l, "c": c, "v": v}


def detector_args(fn):
    return [arrays[p] for p in inspect.signature(fn).parameters if p in arrays]


timings = {}
for name in tap.list_all_patterns():
    fn = getattr(tap, name, None) or getattr(cp, name, None)
    if not callable(fn):
        continue
    args = detector_args(fn)
    tap.clear_cache()
    timings[name] = min(timed(lambda fn=fn, args=args: fn(*args))
                        for _ in range(2)) * 1000

per = pd.Series(timings, name="ms").sort_values(ascending=False)
print(f"{len(per)} detectors, {per.sum():.0f} ms summed with the cache cleared "
      f"between each\nmedian {per.median():.2f} ms, "
      f"top 10 = {per.head(10).sum() / per.sum():.0%} of the total")
per.head(10).round(2).to_frame()

298 detectors, 159 ms summed with the cache cleared between each
median 0.10 ms, top 10 = 25% of the total


,ms
v_top,4.26
mountain,4.25
v_bottom,4.13
three_valleys,4.01
three_peaks,4.00
three_peaks_domed_house,3.96
carl_v,3.95
two_b,3.93
failure_swing,3.92
key_reversal,3.81


## 9. What does not scale

Most detectors are linear in the number of bars. A handful are not: they search
*pairs and triples of pivots*, and the pivot count itself grows with the series,
so their cost grows roughly with its square.

Comparing per-detector cost at two lengths isolates them. A value of `1.0x`
means perfectly linear; `5.0x` means five times worse than linear.

In [23]:
SMALL, LARGE = 1_000, 8_000

cost = {}
for n in (SMALL, LARGE):
    a = synthetic(n, seed=42)
    arrs = {"o": a.open, "h": a.high, "l": a.low, "c": a.close, "v": a.volume}
    for name in tap.list_all_patterns():
        fn = getattr(tap, name, None) or getattr(cp, name, None)
        if not callable(fn):
            continue
        args = [arrs[p] for p in inspect.signature(fn).parameters if p in arrs]
        tap.clear_cache()
        cost.setdefault(name, {})[n] = timed(lambda fn=fn, args=args: fn(*args))

scaling = pd.DataFrame(cost).T * 1000
scaling.columns = [f"{SMALL // 1000}k ms", f"{LARGE // 1000}k ms"]
scaling["vs linear"] = (scaling.iloc[:, 1] / scaling.iloc[:, 0]
                        / (LARGE / SMALL)).round(1)

total_small, total_large = scaling.iloc[:, 0].sum(), scaling.iloc[:, 1].sum()
print(f"summed detector cost: {total_small:.0f} ms at {SMALL:,} bars -> "
      f"{total_large:.0f} ms at {LARGE:,} bars "
      f"({total_large / total_small / (LARGE / SMALL):.2f}x worse than linear)")

worst = scaling[scaling.iloc[:, 0] > 0.2].sort_values("vs linear",
                                                      ascending=False).head(8)
print(f"these 8 are {worst.iloc[:, 1].sum() / total_large:.0%} "
      f"of the cost at {LARGE:,} bars")
worst.round(2)

summed detector cost: 144 ms at 1,000 bars -> 1346 ms at 8,000 bars (1.17x worse than linear)
these 8 are 26% of the cost at 8,000 bars


,1k ms,8k ms,vs linear
abc_correction,1.55,87.31,7.1
measured_move_up,0.99,55.44,7.0
measured_move_down,1.03,56.24,6.8
hs_bottom,0.88,28.85,4.1
busted_hs_bottom,0.96,28.20,3.7
complex_hs_bottom,1.42,34.12,3.0
complex_hs_top,1.40,33.97,3.0
busted_hs_top,1.33,29.65,2.8


These are the head-and-shoulders family, the measured moves and the ABC
correction. If you scan decades of daily data, or intraday bars, either cap
their `window` or exclude them by name:

```python
costly = ("hs_", "busted_hs_", "complex_hs_", "measured_move_", "abc_")
cheap = [p for p in tap.list_all_patterns(module="chart")
         if not p.startswith(costly)]

X = tap.batch_all(o, h, l, c, v=v, chart_patterns=cheap)
```

`chart_patterns=` only accepts chart names, hence `module="chart"` — passing
the full catalogue raises `ValueError`. Everything else is linear, and a
300-detector scan of a few thousand bars stays well under a second.

## 10. A feature frame for a model

Putting it together: drop the columns that barely fire, add the aggregate score,
attach a forward-return target, and you have a modelling frame.

In [24]:
keep = X.loc[:, (X != 0).sum() >= 5]        # drop near-constant columns

feat = pd.concat([df[["close", "volume"]], keep, score], axis=1)
feat["target_fwd_5d"] = df.close.pct_change(5).shift(-5)
feat = feat.dropna()

print(f"{feat.shape[0]} rows x {feat.shape[1]} columns "
      f"({keep.shape[1]} pattern columns kept of {X.shape[1]})")
print(f"memory: {feat.memory_usage(deep=True).sum() / 1e6:.2f} MB")

feat[["close", "net_score", *keep.columns[:5], "target_fwd_5d"]].tail(5).round(4)

1249 rows x 217 columns (213 pattern columns kept of 300)
memory: 0.31 MB


,close,net_score,above_stomach,advance_block,below_stomach,belt_hold_bearish,belt_hold_bullish,target_fwd_5d
date,,,,,,,,
2026-09-08,316.22,-4,0,0,0,0,0,0.0478
2026-09-09,315.34,-1,0,0,0,0,0,0.0541
2026-09-10,326.57,16,0,0,0,0,1,0.0319
2026-09-11,332.27,5,0,0,0,0,0,0.0116
2026-09-14,333.08,0,0,0,0,0,0,0.0177


One last caveat, and it is the important one. The correlations below are
**in-sample, on a single symbol, with no multiple-testing correction**. With 213
columns against one target, the largest values here are what you would expect
from noise alone. They show the mechanics of using the output, not evidence that
any pattern predicts anything.

In [25]:
corr = feat[list(keep.columns)].corrwith(feat.target_fwd_5d).dropna()
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(8)

pd.DataFrame({"corr with fwd 5d": top.round(4),
              "fires": [int((feat[n] != 0).sum()) for n in top.index]})

,corr with fwd 5d,fires
falling_volume_trend,0.1357,552
rising_volume_trend,0.1111,521
roof,-0.1106,22
rounding_top,-0.0996,15
broadening_wedge_desc,0.0972,30
falling_wedge,0.0972,30
cat_ears,-0.0904,39
volume_breakout_day,0.0893,24


## Where to go next

- **[Pattern catalog](../docs/patterns_catalog.md)** — all 300 detectors with
  their parameters
- **[Core concepts](../docs/concepts.md)** — pivots, confirmation delay, and
  the point-in-time guarantee
- **[API reference](../docs/api_reference.md)** — every scanner and helper
- `tap.list_all_patterns(direction="bullish", module="chart")` — filter the
  catalog from code